# Qwen3 Embedding 8B Fine-Tuning (768 dim)

This notebook fine-tunes Qwen3-Embedding-8B on pre-generated curriculum pairs and evaluates on fixed test pairs.
It uses standard SentenceTransformer training (no LoRA/PEFT).

Dataset: data_new/curriculum_training_pairs_complete.json + data_new/fixed_test_pairs.json
Base model: Qwen/Qwen3-Embedding-8B
Target embedding dim: 768

## ⚠️ MEMORY REQUIREMENTS

**Qwen3-8B requires ~10-12GB free GPU memory to load.**

If you get OOM errors:
1. **Restart Jupyter kernel** - Kernel > Restart Kernel
2. **Close other notebooks** that are using GPU memory
3. **Alternative model**: Change `model_name` to `sentence-transformers/all-mpnet-base-v2` (only 420MB)
4. **Use CPU**: Change device detection to force CPU (slower but works)

Check GPU usage before running:
```python
# In a new cell:
import torch
if torch.cuda.is_available():
    print(f"Free: {torch.cuda.mem_get_info()[0]/1e9:.2f}GB")
    print(f"Total: {torch.cuda.mem_get_info()[1]/1e9:.2f}GB")
```


In [9]:
import json
import random
import logging
from pathlib import Path
from datetime import datetime
import gc
import os

import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    roc_curve,
)
from scipy.stats import spearmanr, pearsonr

from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import SentenceEvaluator

import matplotlib.pyplot as plt
import seaborn as sns

# Set environment variable for better CUDA memory management
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


In [10]:
# Memory Check - Run this first to see available GPU memory
import torch
import gc

if torch.cuda.is_available():
    # Clean up any existing allocations
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    
    free_mem = torch.cuda.mem_get_info()[0] / 1e9
    total_mem = torch.cuda.mem_get_info()[1] / 1e9
    used_mem = total_mem - free_mem
    
    print("="*60)
    print("GPU MEMORY STATUS")
    print("="*60)
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total: {total_mem:.2f}GB")
    print(f"Used: {used_mem:.2f}GB ({used_mem/total_mem*100:.1f}%)")
    print(f"Free: {free_mem:.2f}GB ({free_mem/total_mem*100:.1f}%)")
    print("="*60)
    
    if free_mem < 10:
        print("\n⚠️  WARNING: Insufficient free memory for Qwen3-8B!")
        print(f"   Need: ~10GB")
        print(f"   Have: {free_mem:.2f}GB")
        print("\n💡 SOLUTIONS:")
        print("   1. Restart kernel (Kernel > Restart Kernel)")
        print("   2. Close other GPU notebooks")
        print("   3. Use MPNet instead (change model_name below)")
    else:
        print(f"\n✅ Sufficient memory ({free_mem:.2f}GB) for Qwen3-8B")
else:
    print("No CUDA device available - will use CPU")


GPU MEMORY STATUS
GPU: NVIDIA GeForce RTX 5090
Total: 34.19GB
Used: 17.96GB (52.5%)
Free: 16.23GB (47.5%)

✅ Sufficient memory (16.23GB) for Qwen3-8B


In [11]:
logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.info


def set_seed(seed: int) -> None:
    """Set random seed for reproducibility across all backends."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)


def get_device():
    """Detect available device and return appropriate settings."""
    if torch.cuda.is_available():
        device = "cuda"
        log(f"[DEVICE] CUDA: {torch.cuda.get_device_name(0)}")
        log(f"[MEMORY] GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        device = "mps"
        log("[DEVICE] MPS (Apple Silicon)")
    else:
        device = "cpu"
        log("[DEVICE] CPU")
    return device


DEVICE = get_device()


[DEVICE] CUDA: NVIDIA GeForce RTX 5090
[MEMORY] GPU: 34.19 GB


In [12]:
CONFIG = {
    "model_name": "Qwen/Qwen3-Embedding-4B",
    "trust_remote_code": True,
    "output_dir": "models/qwen3_embedding_4b_768",
    "train_pairs_path": "data_new/curriculum_training_pairs_complete.json",
    "test_pairs_path": "data_new/fixed_test_pairs.json",
    "use_curriculum": True,
    "epochs": 12,
    "epochs_per_phase": 4,
    "batch_size": 8,  # Reduced for 4B model safety
    "lr": 5e-5,
    "warmup_ratio": 0.1,
    "max_seq_length": 256,
    "embedding_dim": 768,
    "seed": 42,
}

set_seed(CONFIG["seed"])
log(f"[CONFIG] Model: {CONFIG['model_name']}")
log(f"[CONFIG] Output: {CONFIG['output_dir']}")
log(f"[CONFIG] Embedding dim: {CONFIG['embedding_dim']}")
log(f"[CONFIG] Batch size: {CONFIG['batch_size']} (optimized for 8B model)")
log(f"[CONFIG] Curriculum learning: {CONFIG['use_curriculum']}")


[CONFIG] Model: Qwen/Qwen3-Embedding-4B
[CONFIG] Output: models/qwen3_embedding_4b_768
[CONFIG] Embedding dim: 768
[CONFIG] Batch size: 8 (optimized for 8B model)
[CONFIG] Curriculum learning: True


In [13]:
def load_pairs(path: str):
    """Load training/test pairs with error handling and validation."""
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except FileNotFoundError:
        raise FileNotFoundError(f"Data file not found: {path}")
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON in {path}: {e}")
    
    # Validate required fields
    required_fields = ["texts1", "texts2", "labels"]
    missing_fields = [f for f in required_fields if f not in data]
    if missing_fields:
        raise ValueError(f"Missing required fields in {path}: {missing_fields}")
    
    texts1 = data["texts1"]
    texts2 = data["texts2"]
    labels = data["labels"]
    phase_indicators = data.get("phase_indicators")
    
    # Validate data consistency
    if not (len(texts1) == len(texts2) == len(labels)):
        raise ValueError(f"Inconsistent data lengths: texts1={len(texts1)}, texts2={len(texts2)}, labels={len(labels)}")
    
    examples = [
        InputExample(texts=[t1, t2], label=float(y))
        for t1, t2, y in zip(texts1, texts2, labels)
    ]
    
    log(f"[LOAD] {path}: {len(examples)} pairs loaded")
    return examples, phase_indicators, data.get("metadata", {})


train_examples, phase_indicators, train_meta = load_pairs(CONFIG["train_pairs_path"])
log(f"[DATA] Train pairs: {len(train_examples)}")

if CONFIG["use_curriculum"] and phase_indicators:
    # Optimized phase splitting using list comprehensions
    phase1 = [ex for ex, p in zip(train_examples, phase_indicators) if p == 1]
    phase2 = [ex for ex, p in zip(train_examples, phase_indicators) if p == 2]
    phase3 = [ex for ex, p in zip(train_examples, phase_indicators) if p == 3]
    
    CURRICULUM_PHASES = {"phase1": phase1, "phase2": phase2, "phase3": phase3}
    log(f"[DATA] Phase sizes: {len(phase1)}, {len(phase2)}, {len(phase3)}")
    
    # Validate phases
    total_phased = len(phase1) + len(phase2) + len(phase3)
    if total_phased != len(train_examples):
        log(f"[WARNING] Phase mismatch: {total_phased} phased vs {len(train_examples)} total")
else:
    CURRICULUM_PHASES = None
    log("[DATA] Curriculum learning disabled or no phase indicators found")


eval_examples, _, _ = load_pairs(CONFIG["test_pairs_path"])
log(f"[DATA] Eval pairs: {len(eval_examples)}")


[LOAD] data_new/curriculum_training_pairs_complete.json: 15000 pairs loaded
[DATA] Train pairs: 15000
[DATA] Phase sizes: 5000, 5000, 5000
[LOAD] data_new/fixed_test_pairs.json: 1000 pairs loaded
[DATA] Eval pairs: 1000


In [14]:
def cleanup_memory():
    """Aggressively clean up GPU memory."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        log("[MEMORY] CUDA cache cleared")


def init_model(config, device):
    """Initialize model with memory-efficient loading."""
    log(f"[MODEL] Loading {config['model_name']}")
    
    # Clean memory before loading
    cleanup_memory()
    
    if torch.cuda.is_available():
        mem_free = torch.cuda.mem_get_info()[0] / 1e9
        mem_total = torch.cuda.mem_get_info()[1] / 1e9
        log(f"[MEMORY] Free GPU memory: {mem_free:.2f}GB / {mem_total:.2f}GB")
        
        if mem_free < 10:  # Less than 10GB free
            log(f"[WARNING] Low GPU memory ({mem_free:.2f}GB free). Consider:")
            log("  1. Restart kernel and run only this notebook")
            log("  2. Use smaller model (MPNet-base instead of Qwen3-8B)")
            log("  3. Use CPU training (slower but more memory)")
            raise RuntimeError(f"Insufficient GPU memory: {mem_free:.2f}GB free, need ~10GB for Qwen3-8B")
    
    # Load model with trust_remote_code
    try:
        model = SentenceTransformer(
            config["model_name"],
            device=device,
            trust_remote_code=config.get("trust_remote_code", False),
        )
        model.max_seq_length = min(config["max_seq_length"], model.max_seq_length)
        
        if torch.cuda.is_available():
            allocated = torch.cuda.memory_allocated() / 1e9
            reserved = torch.cuda.memory_reserved() / 1e9
            log(f"[MEMORY] Model loaded: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved")
        
        return model
    
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            log(f"[ERROR] GPU OOM while loading model: {e}")
            log("\n[SOLUTION] Try one of these:")
            log("  1. Restart kernel: Kernel > Restart Kernel")
            log("  2. Switch to MPNet: Change model_name to 'sentence-transformers/all-mpnet-base-v2'")
            log("  3. Use CPU: Change DEVICE to 'cpu' (will be slower)")
            log("  4. Close other GPU processes")
            raise
        else:
            raise


log("\n" + "="*60)
log("[MODEL INITIALIZATION]")
log("="*60)

# Clean memory before model loading
cleanup_memory()

model = init_model(CONFIG, DEVICE)
log("[MODEL] Initialization complete")
log("="*60)



[MODEL INITIALIZATION]
[MEMORY] CUDA cache cleared
[MODEL] Loading Qwen/Qwen3-Embedding-4B
[MEMORY] CUDA cache cleared
[MEMORY] Free GPU memory: 16.23GB / 34.19GB
Load pretrained SentenceTransformer: Qwen/Qwen3-Embedding-4B


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

1 prompt is loaded, with the key: query
[MEMORY] Model loaded: 32.40GB allocated, 32.40GB reserved
[MODEL] Initialization complete


In [15]:
class ITSMEvaluator(SentenceEvaluator):
    def __init__(self, examples, batch_size=32, embedding_dim=None, name="eval"):
        self.examples = examples
        self.batch_size = batch_size
        self.embedding_dim = embedding_dim
        self.name = name

    def __call__(self, model, output_path=None, epoch=-1, steps=-1):
        metrics = evaluate_pairs(model, self.examples, self.batch_size, self.embedding_dim)
        log(
            f"[EVAL] {self.name} epoch={epoch} spearman={metrics['spearman']:.4f} "
            f"roc_auc={metrics['roc_auc']:.4f} f1={metrics['f1']:.4f}"
        )
        return metrics["spearman"]


def encode_texts(model, texts, batch_size=32, truncate_dim=None):
    """Encode texts with optional dimension truncation."""
    kwargs = dict(batch_size=batch_size, show_progress_bar=True, convert_to_numpy=True)
    if truncate_dim:
        try:
            return model.encode(texts, truncate_dim=truncate_dim, **kwargs)
        except TypeError:
            embeddings = model.encode(texts, **kwargs)
            return embeddings[:, :truncate_dim]
    return model.encode(texts, **kwargs)


def evaluate_pairs(model, examples, batch_size=32, embedding_dim=None):
    """Evaluate model on pairs with comprehensive metrics."""
    texts1 = [ex.texts[0] for ex in examples]
    texts2 = [ex.texts[1] for ex in examples]
    labels = np.array([ex.label for ex in examples])

    emb1 = encode_texts(model, texts1, batch_size=batch_size, truncate_dim=embedding_dim)
    emb2 = encode_texts(model, texts2, batch_size=batch_size, truncate_dim=embedding_dim)
    scores = np.sum(emb1 * emb2, axis=1) / (
        np.linalg.norm(emb1, axis=1) * np.linalg.norm(emb2, axis=1) + 1e-8
    )

    spearman = spearmanr(labels, scores).correlation
    pearson = pearsonr(labels, scores).correlation
    roc_auc = roc_auc_score(labels, scores)
    fpr, tpr, _ = roc_curve(labels, scores)
    pr_auc = average_precision_score(labels, scores)

    precision, recall, thresholds = precision_recall_curve(labels, scores)
    f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
    best_idx = int(np.argmax(f1_scores))
    best_threshold = thresholds[best_idx - 1] if best_idx > 0 else 0.5

    preds = (scores >= best_threshold).astype(int)
    f1 = f1_score(labels, preds)
    accuracy = accuracy_score(labels, preds)
    precision_at = precision_score(labels, preds)
    recall_at = recall_score(labels, preds)
    cm = confusion_matrix(labels, preds).tolist()

    return {
        "scores": scores.tolist(),
        "labels": labels.tolist(),
        "precision_curve": precision.tolist(),
        "recall_curve": recall.tolist(),
        "thresholds": thresholds.tolist(),
        "f1_curve": f1_scores.tolist(),
        "roc_fpr": fpr.tolist(),
        "roc_tpr": tpr.tolist(),
        "spearman": float(spearman),
        "pearson": float(pearson),
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
        "f1": float(f1),
        "accuracy": float(accuracy),
        "precision": float(precision_at),
        "recall": float(recall_at),
        "best_threshold": float(best_threshold),
        "confusion_matrix": cm,
    }


In [16]:
import gc

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
save_path = Path(CONFIG["output_dir"]) / f"qwen3_4b_768_{timestamp}"
# Convert to absolute path to avoid Windows mkdir issues
save_path = save_path.resolve()
save_path.mkdir(parents=True, exist_ok=True)

train_loss = losses.MultipleNegativesRankingLoss(model)

# Initialize evaluator with correct batch size and embedding_dim
evaluator = ITSMEvaluator(
    eval_examples, 
    batch_size=CONFIG["batch_size"], 
    embedding_dim=CONFIG["embedding_dim"],
    name="eval"
)
use_amp = DEVICE == "cuda"

log(f"[TRAIN] Starting curriculum training with {len(CURRICULUM_PHASES) if CURRICULUM_PHASES else 0} phases")
log(f"[TRAIN] Output path: {save_path}")

if CONFIG["use_curriculum"] and CURRICULUM_PHASES:
    for phase_idx, (phase_name, phase_examples) in enumerate(sorted(CURRICULUM_PHASES.items()), 1):
        log(f"\n{'='*60}")
        log(f"[PHASE {phase_idx}/3] {phase_name.upper()}")
        log(f"{'='*60}")
        
        phase_loader = DataLoader(
            phase_examples,
            batch_size=CONFIG["batch_size"],
            shuffle=True,
            num_workers=0,
            pin_memory=(DEVICE == "cuda"),
        )
        total_steps = len(phase_loader) * CONFIG["epochs_per_phase"]
        warmup_steps = int(total_steps * CONFIG["warmup_ratio"])

        log(f"[TRAIN] Examples: {len(phase_examples)}, Epochs: {CONFIG['epochs_per_phase']}")
        log(f"[TRAIN] Steps: {total_steps}, Warmup: {warmup_steps}")
        
        model.fit(
            train_objectives=[(phase_loader, train_loss)],
            evaluator=evaluator,
            epochs=CONFIG["epochs_per_phase"],
            warmup_steps=warmup_steps,
            optimizer_params={"lr": CONFIG["lr"]},
            scheduler="WarmupCosine",
            output_path=str(save_path),
            evaluation_steps=max(100, len(phase_loader) // 2),
            save_best_model=True,
            show_progress_bar=True,
            use_amp=use_amp,
        )
        
        # Memory cleanup between phases
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
            if torch.cuda.is_available():
                allocated = torch.cuda.memory_allocated() / 1e9
                reserved = torch.cuda.memory_reserved() / 1e9
                log(f"[MEMORY] GPU after phase {phase_idx}: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved")
        elif DEVICE == "mps":
            torch.mps.empty_cache()
            log(f"[MEMORY] MPS cache cleared after phase {phase_idx}")
        
else:
    train_loader = DataLoader(
        train_examples,
        batch_size=CONFIG["batch_size"],
        shuffle=True,
        num_workers=0,
        pin_memory=(DEVICE == "cuda"),
    )
    total_steps = len(train_loader) * CONFIG["epochs"]
    warmup_steps = int(total_steps * CONFIG["warmup_ratio"])
    
    log(f"[TRAIN] Standard training (no curriculum)")
    log(f"[TRAIN] Examples: {len(train_examples)}, Epochs: {CONFIG['epochs']}")
    
    model.fit(
        train_objectives=[(train_loader, train_loss)],
        evaluator=evaluator,
        epochs=CONFIG["epochs"],
        warmup_steps=warmup_steps,
        optimizer_params={"lr": CONFIG["lr"]},
        scheduler="WarmupCosine",
        output_path=str(save_path),
        evaluation_steps=max(100, len(train_loader) // 2),
        save_best_model=True,
        show_progress_bar=True,
        use_amp=use_amp,
    )

log(f"\n[TRAIN] Complete. Best model saved to: {save_path}")


FileNotFoundError: [WinError 2] The system cannot find the file specified: 'models\\qwen3_embedding_4b_768\\qwen3_4b_768_20260101_0013'

In [ ]:
log("\n[FINAL EVALUATION]")
log("="*60)

# Load best model with trust_remote_code
best_model = SentenceTransformer(
    str(save_path), 
    device=DEVICE,
    trust_remote_code=CONFIG["trust_remote_code"]
)

# Evaluate with correct parameters
eval_metrics = evaluate_pairs(
    best_model, 
    eval_examples, 
    batch_size=CONFIG["batch_size"],
    embedding_dim=CONFIG["embedding_dim"]
)

log(f"[FINAL] Spearman: {eval_metrics['spearman']:.4f}")
log(f"[FINAL] Pearson: {eval_metrics['pearson']:.4f}")
log(f"[FINAL] ROC-AUC: {eval_metrics['roc_auc']:.4f}")
log(f"[FINAL] PR-AUC: {eval_metrics['pr_auc']:.4f}")
log(f"[FINAL] F1: {eval_metrics['f1']:.4f}")
log(f"[FINAL] Accuracy: {eval_metrics['accuracy']:.4f}")
log(f"[FINAL] Precision: {eval_metrics['precision']:.4f}")
log(f"[FINAL] Recall: {eval_metrics['recall']:.4f}")
log(f"[FINAL] Best threshold: {eval_metrics['best_threshold']:.4f}")
log(f"[FINAL] Confusion matrix: {eval_metrics['confusion_matrix']}")
log("="*60)


In [ ]:
# Plots: ROC, PR, confusion matrix, score distribution, F1 vs threshold
scores = np.array(eval_metrics["scores"])
labels = np.array(eval_metrics["labels"])

# ROC curve
plt.figure(figsize=(6, 5))
plt.plot(eval_metrics["roc_fpr"], eval_metrics["roc_tpr"], label=f"ROC AUC={eval_metrics['roc_auc']:.3f}")
plt.plot([0, 1], [0, 1], "k--", alpha=0.5)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

# PR curve
plt.figure(figsize=(6, 5))
plt.plot(eval_metrics["recall_curve"], eval_metrics["precision_curve"], label=f"PR AUC={eval_metrics['pr_auc']:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend(loc="lower left")
plt.tight_layout()
plt.show()

# Confusion matrix
cm = np.array(eval_metrics["confusion_matrix"])
plt.figure(figsize=(4.5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

# Score distribution
plt.figure(figsize=(6, 5))
plt.hist(scores[labels == 0], bins=50, alpha=0.6, label="Negative")
plt.hist(scores[labels == 1], bins=50, alpha=0.6, label="Positive")
plt.xlabel("Similarity Score")
plt.ylabel("Count")
plt.title("Score Distribution")
plt.legend()
plt.tight_layout()
plt.show()

# F1 vs threshold
thr = np.array(eval_metrics["thresholds"])
f1_curve = np.array(eval_metrics["f1_curve"])
if len(f1_curve) == len(thr) + 1:
    f1_plot = f1_curve[1:]
else:
    f1_plot = f1_curve[:len(thr)]
plt.figure(figsize=(6, 5))
plt.plot(thr, f1_plot)
plt.axvline(eval_metrics["best_threshold"], color="r", linestyle="--", label=f"Best={eval_metrics['best_threshold']:.3f}")
plt.xlabel("Threshold")
plt.ylabel("F1")
plt.title("F1 vs Threshold")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
metadata = {
    "timestamp": datetime.now().isoformat(),
    "model_name": CONFIG["model_name"],
    "output_dir": str(save_path),
    "train_pairs_path": CONFIG["train_pairs_path"],
    "test_pairs_path": CONFIG["test_pairs_path"],
    "embedding_dim": CONFIG["embedding_dim"],
    "training": {
        "epochs": CONFIG["epochs"],
        "epochs_per_phase": CONFIG["epochs_per_phase"],
        "batch_size": CONFIG["batch_size"],
        "lr": CONFIG["lr"],
        "warmup_ratio": CONFIG["warmup_ratio"],
        "max_seq_length": CONFIG["max_seq_length"],
        "use_curriculum": CONFIG["use_curriculum"],
    },
    "device": DEVICE,
    "metrics": eval_metrics,
}

metadata_path = Path(save_path) / "training_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

log(f"[OK] Metadata saved to {metadata_path}")